<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zeshengliu213/ai-for-materials/blob/main/08%20-%20Sequence%20Models%20and%20Transformers/08b_smiles_lstm_aqsoldb.ipynb)

# 08b · 进阶序列建模：数据清洗、SMILES 增强、似然评估与跨领域迁移微调

在 `08a` 中，我们在 3 万个结构高度同质的哈佛清洁能源计划（CEP）有机光伏分子上从头训练了 SMILES 语言模型。但在真实的材料与化学研发中，我们经常面临两个棘手的工程问题：

1. **通用数据库良莠不齐**：像 AqSolDB 这样的实验数据库里混杂着多组分盐、金属配合物、带电离子和超长聚合物片段，直接塞给语言模型会导致词表膨胀与垃圾输出；
2. **目标材料领域样本极少**：假设课题组正在研发新型有机半导体（如并苯、寡聚噻吩、BTBT 稠环体系），手里收集到的高质量文献分子往往只有一百多个（`data/osc_finetune.csv` 共 159 个分子）。用一百多个分子从头训练神经网络，连基本的括号和闭环语法都学不会。

如何解决“小样本领域分子生成”？Segler 等人（*ACS Cent. Sci.* 2018）提出了分子生成领域经典的**预训练—微调（Pretraining & Fine-Tuning）**范式：先在数千至数万个通用分子上预训练语言模型，让它充分掌握 SMILES 语法与基础化学价态；再结合 **SMILES 随机重写增强（SMILES Enumeration）**，在百余个目标领域分子上微调几轮，即可将生成分布快速迁移到目标材料家族。

**本节核心内容：**
1. 异质化学库的数据清洗、长度截断与面向对象词表封装（`Vocab`）
2. 通用分子库基线训练与 **SMILES 增强（SMILES Augmentation）** 的作用机制
3. 化学空间 PCA 投影检验与**每词元负对数似然（NLL per token）**：衡量分子在模型眼中的“自然度”
4. 跨领域迁移学习：将 AqSolDB 预训练模型微调至 135 个有机半导体分子，观察泛化与**灾难性遗忘（Catastrophic Forgetting）**
5. 生成器与性质预测模型的串联虚拟筛选及架构演进展望

> **参考答案提示**：本节课内练习（`练习 08b-1` 至 `08b-4`）的完整代码与分析见同目录下的 `solutions/08_sequence_solutions.ipynb`。

In [ ]:
# colab-setup (本地运行时自动跳过)
import os, sys

if "google.colab" in sys.modules:
    _repo_dir = "/content/ai-for-materials"
    if not os.path.exists(_repo_dir):
        os.system("git clone --depth 1 https://github.com/Zeshengliu213/ai-for-materials.git " + _repo_dir)
    os.chdir(os.path.join(_repo_dir, "08 - Sequence Models and Transformers"))
    get_ipython().run_line_magic("pip", "install -q rdkit")


In [ ]:
try:
    import rdkit
except ImportError:
    %pip install -q rdkit
    import rdkit

import os
import re
import copy
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors, Draw

RDLogger.DisableLog("rdApp.*")

def set_seed(seed=0):
    torch.manual_seed(seed)
    np.random.seed(seed)
    torch.use_deterministic_algorithms(True)

set_seed(0)
print("PyTorch version:", torch.__version__)

<!-- ai4m-style -->
> **📘 课程统一绘图风格**：下面这格加载课程自带的 `ai4m.mplstyle`（配色与课程网站同源，无上右边框，仅横向浅网格，保存默认 300 dpi）。`C` 是课程配色，手动指定颜色时写 `color=C.crimson`。


In [ ]:
# ai4m-style：课程统一绘图风格（本地与 Colab 通用）
import pathlib, sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))   # course/ 根目录
import ai4m_style as ai4m

C = ai4m.use()        # 应用 ai4m.mplstyle，返回课程配色


In [ ]:
def find_file(name):
    for path in (f"data/{name}", name):
        if os.path.exists(path):
            return path
    try:
        from google.colab import files
        print(f"请上传 {name}")
        files.upload()
        return name
    except ImportError:
        raise FileNotFoundError(f"找不到 {name}，请确保它位于 data/ 目录下")

df = pd.read_csv(find_file("solubility.csv"))
print(f"成功读取 AqSolDB 数据集，共 {len(df)} 条记录")

---
## 1. 异质化学库的数据清洗与词表封装

### 1.1 过滤盐、离子与生僻元素

`08a` 的 CEP 数据集是人工生成的虚拟分子库，所有分子格式统一；而真实的 AqSolDB 实验水溶解度数据库里包含了大量无机盐（`[Na+].[Cl-]`）、有机金属配合物、带电离域结构以及上百个原子的长链表面活性剂。

由于生成模型会忠实模仿训练集中出现的一切特征，在训练通用中性有机小分子生成模型之前，必须先建立严格的**化学清洗流水线（Data Curation Pipeline）**，只保留：

1. **单一连通组分**（SMILES 中不含片段分隔符 `.`）；
2. **电中性分子**（所有原子的形式电荷 `GetFormalCharge()` 均为 0）；
3. **常见有机与卤素元素**（仅含 `C, N, O, S, F, Cl, Br, I, P`）；
4. **适中的分子尺寸**（重原子数在 $5 \sim 50$ 之间），并统一转换为 RDKit 规范 SMILES 去重。

In [ ]:
ALLOWED = {"C", "N", "O", "S", "F", "Cl", "Br", "I", "P"}

def clean(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None or "." in smi:
        return None
    if any(a.GetFormalCharge() != 0 for a in mol.GetAtoms()):
        return None
    if any(a.GetSymbol() not in ALLOWED for a in mol.GetAtoms()):
        return None
    if not 5 <= mol.GetNumHeavyAtoms() <= 50:
        return None
    return Chem.MolToSmiles(mol)

cleaned = [clean(s) for s in df["SMILES"]]
removed = [s for s, c in zip(df["SMILES"], cleaned) if c is None and len(s) < 40]
print("被过滤掉的典型分子示例:", removed[:8])
smiles_all = sorted(set(c for c in cleaned if c))
print(f"\n清洗并去重后剩余 {len(smiles_all)} 个分子")

### 1.2 面向对象词表封装（`Vocab`）与序列长度截断

为了在预训练、似然评估和迁移微调阶段方便地复用分词与编解码逻辑，我们将正则分词器与词表操作封装为一个轻量的 `Vocab` 类。

同时要注意矩阵补齐（padding）带来的计算开销：同一个批次的分子矩阵宽度由最长序列决定，如果保留极少数长度为 90–100 的长链分子，绝大多数短分子后面都会挂上几十个无意义的 `<pad>`。观察下方 AqSolDB 的词元长度分布，95% 的分子长度不超过 50 个词元，因此我们在 60 个词元处截断（加上首尾的 `^` 和 `$` 后固定矩阵宽度 `MAXLEN = 62`）。截断极长分子后，几个仅在复杂天然产物中出现过一两次的生僻立体化学词元（如 `[P@@]`、`[PH]`）也随之自然剔除，使词表收敛到紧凑的 **37 个词元**。

In [ ]:
TOKEN_RE = re.compile(r"(\[[^\]]+\]|Br|Cl|%\d{2}|.)")

def tokenize(smi):
    return TOKEN_RE.findall(smi)

c8btbt = "CCCCCCCCc1ccc2c(c1)sc1c2sc2cc(CCCCCCCC)ccc12"
print("C8-BTBT 分词结果:", tokenize(c8btbt))

class Vocab:
    def __init__(self, tokens):
        self.tokens = list(tokens)
        self.stoi = {t: i for i, t in enumerate(self.tokens)}
        self.pad, self.bos, self.eos = self.stoi["<pad>"], self.stoi["^"], self.stoi["$"]

    def __len__(self):
        return len(self.tokens)

    def encode(self, smi):
        return [self.bos] + [self.stoi[t] for t in tokenize(smi)] + [self.eos]

    def decode(self, ids):
        out = []
        for i in ids:
            if i == self.eos:
                break
            if i not in (self.pad, self.bos):
                out.append(self.tokens[i])
        return "".join(out)

def build_vocab(smiles_list):
    toks = set()
    for s in smiles_list:
        toks.update(tokenize(s))
    return Vocab(["<pad>", "^", "$"] + sorted(toks))

In [ ]:
lengths = np.array([len(tokenize(s)) for s in smiles_all])
print(f"词元长度统计: 中位数 {int(np.median(lengths))} | 95% 分位数 {int(np.percentile(lengths, 95))} | 最大值 {lengths.max()}")

smiles_all = [s for s in smiles_all if len(tokenize(s)) <= 60]
MAXLEN = 62                                  # 最多 60 个化学词元 + ^ + $
vocab = build_vocab(smiles_all)
print(f"截断长度 > 60 的分子后剩余 {len(smiles_all)} 个分子，最终词表大小: {len(vocab)}")
print("词表列表:", vocab.tokens)

fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(lengths, bins=60)
ax.axvline(60, color="tab:red", ls="--", label="cutoff = 60")
ax.set_xlabel("Number of tokens")
ax.set_ylabel("Molecules")
ax.set_title("SMILES token length distribution in AqSolDB")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
def to_tensor(smiles_list, vocab, maxlen=MAXLEN):
    X = torch.full((len(smiles_list), maxlen), vocab.pad, dtype=torch.long)
    for i, s in enumerate(smiles_list):
        ids = vocab.encode(s)
        X[i, : len(ids)] = torch.tensor(ids)
    return X

rng = np.random.default_rng(0)
perm = rng.permutation(len(smiles_all))
val_smiles = [smiles_all[i] for i in perm[:500]]
train_smiles = [smiles_all[i] for i in perm[500:]]
train_set = set(train_smiles)

X_train = to_tensor(train_smiles, vocab)
X_val = to_tensor(val_smiles, vocab)
print("训练集张量形状:", X_train.shape, "| 验证集张量形状:", X_val.shape)

### 1.3 追踪张量在 `SmilesLSTM` 中的形状变化

我们在 `SmilesLSTM` 的 `nn.Embedding` 中显式指定 `padding_idx=0`，使 `<pad>` 词元的嵌入向量固定为零向量。下面以溴乙烷 `CCBr`（编码后长度为 5：`^ C C Br $`）为例，逐步验证经过 `Embedding`、双层 `LSTM` 与输出层 `Linear` 后的张量维度：

In [ ]:
class SmilesLSTM(nn.Module):
    def __init__(self, vocab_size, emb=64, hidden=256, layers=2, dropout=0.2):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb, padding_idx=0)
        self.lstm = nn.LSTM(emb, hidden, layers, batch_first=True, dropout=dropout)
        self.out = nn.Linear(hidden, vocab_size)

    def forward(self, x, state=None):
        h, state = self.lstm(self.emb(x), state)
        return self.out(h), state

model = SmilesLSTM(len(vocab))
x = torch.tensor([vocab.encode("CCBr")])
e = model.emb(x)
h, (h_n, c_n) = model.lstm(e)
logits = model.out(h)

print("输入编号张量 x:     ", x.tolist(), "-> 形状", tuple(x.shape))
print("Embedding 输出 e:   ", tuple(e.shape), "(每个词元映射为 64 维向量)")
print("LSTM 隐状态序列 h:  ", tuple(h.shape), "(虽然第 1、2 步输入都是 C，但隐状态融合了不同前文，因此 h[0,1] != h[0,2])")
print("Linear 输出 logits: ", tuple(logits.shape), "(每一步对 37 个候选词元打分)")

---
## 2. 规范 SMILES 基线训练 vs. SMILES 增强预训练模型

我们先在规范 SMILES（Canonical SMILES）训练集 `X_train` 上训练 10 轮作为基准，同时记录每轮在验证集上的**每词元平均负对数似然（Mean NLL per token）**和 200 个采样分子的有效率：

In [ ]:
loss_fn = nn.CrossEntropyLoss(ignore_index=vocab.pad)

@torch.no_grad()
def sample(model, vocab, n=100, temperature=1.0, maxlen=100):
    model.eval()
    x = torch.full((n, 1), vocab.bos, dtype=torch.long)
    state, out = None, []
    finished = torch.zeros(n, dtype=torch.bool)
    for _ in range(maxlen):
        logits, state = model(x, state)
        probs = torch.softmax(logits[:, -1] / temperature, dim=-1)
        x = torch.multinomial(probs, 1)
        out.append(x)
        finished |= x.squeeze(1) == vocab.eos
        if finished.all():
            break
    return [vocab.decode(row) for row in torch.cat(out, dim=1).tolist()]

def canonical(smi):
    mol = Chem.MolFromSmiles(smi)
    return Chem.MolToSmiles(mol) if mol is not None else None

def evaluate_samples(samples, reference_set):
    valid = [c for c in (canonical(s) for s in samples) if c is not None]
    unique = set(valid)
    novel = [s for s in unique if s not in reference_set]
    return {"validity": len(valid) / len(samples),
            "uniqueness": len(unique) / max(1, len(valid)),
            "novelty": len(novel) / max(1, len(unique))}

In [ ]:
@torch.no_grad()
def mean_nll(model, X, vocab, batch_size=256):
    """计算数据集 X 上每个非填充词元的平均负对数似然（NLL per token）。"""
    model.eval()
    total, count = 0.0, 0
    for i in range(0, len(X), batch_size):
        xb = X[i : i + batch_size]
        logits, _ = model(xb[:, :-1])
        total += nn.functional.cross_entropy(logits.reshape(-1, len(vocab)), xb[:, 1:].reshape(-1),
                                             ignore_index=vocab.pad, reduction="sum").item()
        count += (xb[:, 1:] != vocab.pad).sum().item()
    return total / count

def train_lm(model, X_train, X_val, vocab, epochs=10, lr=2e-3, batch_size=128, n_sample=200, verbose=True):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    for ep in range(epochs):
        model.train()
        t0 = time.time()
        perm = torch.randperm(len(X_train))
        total, n_batches = 0.0, 0
        for i in range(0, len(X_train), batch_size):
            xb = X_train[perm[i : i + batch_size]]
            logits, _ = model(xb[:, :-1])
            loss = loss_fn(logits.reshape(-1, len(vocab)), xb[:, 1:].reshape(-1))
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total += loss.item(); n_batches += 1
        val_nll = mean_nll(model, X_val, vocab)
        metrics = evaluate_samples(sample(model, vocab, n=n_sample), train_set)
        history.append({"epoch": ep + 1, "train": total / n_batches, "val": val_nll, **metrics})
        if verbose:
            print(f"Epoch {ep+1:2d} | Train NLL: {total/n_batches:.3f} | Val NLL: {val_nll:.3f} | "
                  f"Validity: {metrics['validity']:.2f} | {time.time()-t0:.0f}s")
    return pd.DataFrame(history)

set_seed(0)
model = SmilesLSTM(len(vocab))
history = train_lm(model, X_train, X_val, vocab, epochs=10)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(history["epoch"], history["train"], "o-", label="train")
axes[0].plot(history["epoch"], history["val"], "s-", label="validation")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("NLL per token"); axes[0].legend(); axes[0].set_title("Loss curve")
axes[1].plot(history["epoch"], history["validity"], "o-", color="tab:green")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Fraction valid"); axes[1].set_ylim(0, 1); axes[1].set_title("Validity of 200 samples")
plt.tight_layout()
plt.show()

### 2.1 加载 SMILES 增强预训练模型（`smiles_lstm_pretrained.pt`）

由于 AqSolDB 清洗后只有 6,814 个训练分子，且结构涵盖脂肪链、多元环、卤代物、酯、酰胺等数十种官能团，只训练 10 轮规范 SMILES 的有效率约为 $60\%\sim 65\%$。

为了在有限的分子数量下让语言模型更透彻地掌握化学图拓扑，我们在预训练权重 `data/smiles_lstm_pretrained.pt`（训练 40 轮）中引入了一项关键技巧——**SMILES 随机重写增强（SMILES Enumeration / Augmentation）**：

- 回忆 `01d` 讲过的分子图遍历：以分子中不同的原子作为起点、按不同的深度优先遍历顺序书写，**同一个分子可以写出许多条合法的非规范 SMILES 字符串**（例如甲苯既可以写成 `Cc1ccccc1`，也可以从环上碳原子出发写成 `c1ccc(C)cc1`）。
- 在每一轮训练开始前，调用 RDKit 的 `Chem.MolToRandomSmilesVect` 为每个训练分子随机生成一条等价的非规范 SMILES。这就像在 `07b` 显微图像分类中对图像做随机旋转一样，强迫模型不要死记硬背某一种固定的字符排列顺序，而是学会底层的环闭合与价态语法。

下面加载该预训练模型并对比它的验证集 NLL 与生成指标：

In [ ]:
ckpt = torch.load(find_file("smiles_lstm_pretrained.pt"), map_location="cpu")
print("Checkpoint 说明:", ckpt["note"])
vocab_pre = Vocab(ckpt["vocab"])
assert vocab_pre.tokens == vocab.tokens, "预训练词表与当前词表不一致"
pretrained = SmilesLSTM(len(vocab_pre), **ckpt["config"])
pretrained.load_state_dict(ckpt["state_dict"])
pretrained.eval()

print(f"规范验证集 NLL/token: 10 轮规范训练模型 = {mean_nll(model, X_val, vocab):.3f} | 40 轮增强预训练模型 = {mean_nll(pretrained, X_val, vocab_pre):.3f}")

set_seed(0)
samples_pre = sample(pretrained, vocab_pre, n=1000)
metrics_pre = evaluate_samples(samples_pre, train_set)
print("40 轮增强预训练模型生成指标 (T=1.0):", {k: round(v, 3) for k, v in metrics_pre.items()})
valid_pre = [c for c in (canonical(s) for s in samples_pre) if c is not None]

观察上面的输出，有一个非常值得深思的现象：**40 轮增强预训练模型的采样有效率（~77%）和唯一率（~99%）明显更高，但它在规范验证集 `X_val` 上的 NLL（0.946）反而比 10 轮模型（0.827）更高！**

为什么更好的生成模型在验证集上的交叉熵损失反而更大？因为 `pretrained` 是在随机重写的非规范 SMILES 上训练的，它把一个分子的总概率质量均匀分摊到了该分子的十几种合法 SMILES 写法上；而 `X_val` 里全是唯一的 RDKit 规范写法。这提醒我们：**负对数似然（NLL）只能在相同的数据表征方式下横向比较**，评价跨表征训练的生成模型时，必须以最终解码出的化学分子有效性、唯一性和新颖性为准。

---
### 练习 08b-1：寻找最大化“有效 + 唯一 + 新颖”分子产量的最优采样温度

使用预训练模型 `pretrained` 在温度 $T \in [0.4, 0.6, 0.8, 1.0, 1.2, 1.4, 1.6]$ 下各采样 500 个分子：
1. 将有效率（`validity`）、唯一率（`uniqueness`）和新颖率（`novelty`）随温度变化的曲线画在同一张图上；
2. 在实际虚拟筛选中，我们关心的是采样 500 次最终能收获多少个**既合法、又不重复、且不在训练集里的新分子**（即三者乘积 $\times 500$）。请计算并打印每个温度下的净收获新分子数量，找出产量最高的温度。

In [ ]:
# 练习 08b-1：在这里写你的代码


### 练习 08b-2：亲手实现 SMILES 随机重写增强训练（`train_lm_augmented`）

下面给出了利用 `Chem.MolToRandomSmilesVect` 随机重写单条 SMILES 的辅助函数 `randomize_smiles_single`。请仿照 `train_lm` 编写一个 `train_lm_augmented` 函数：在每个 epoch 开始时，将 `train_smiles` 中的每个分子随机重写一遍再打包成张量训练（其余超参数与 `train_lm` 完全一致：训练 10 轮，隐藏层 256）。

训练完成后，将每轮的采样有效率（`validity`）和规范验证集损失（`val`）与不增强的 `history` 画在同一张图上对比，验证刚才关于 SMILES 增强的分析。（注：运行 10 轮约需 2–3 分钟。）

In [ ]:
def randomize_smiles_single(smi, vocab, rng, max_tokens=MAXLEN - 2):
    mol = Chem.MolFromSmiles(smi)
    for r in Chem.MolToRandomSmilesVect(mol, 10, randomSeed=int(rng.integers(1 << 31))):
        if len(tokenize(r)) <= max_tokens and all(t in vocab.stoi for t in tokenize(r)):
            return r
    return smi

# 练习 08b-2：在这里写你的代码


---
## 3. 化学空间投影与似然（NLL）异常检测

### 3.1 生成分子在 AqSolDB 化学空间中的分布

我们先对比预训练模型生成的合法分子 `valid_pre` 与 AqSolDB 训练集在分子量（`MolWt`）、脂水分配系数（`MolLogP`）和芳香环数（`NumAromaticRings`）上的分布，再将生成分子投影到第 `03a` 讲建立的 17 维 RDKit 描述符 PCA 二维平面上：

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

def props(smiles_list):
    mols = [Chem.MolFromSmiles(s) for s in smiles_list]
    return pd.DataFrame({"MolWt": [Descriptors.MolWt(m) for m in mols],
                         "MolLogP": [Descriptors.MolLogP(m) for m in mols],
                         "NumAromaticRings": [Descriptors.NumAromaticRings(m) for m in mols]})

props_train = props(train_smiles)
props_gen = props(valid_pre)

feature_cols = ["MolWt", "MolLogP", "MolMR", "HeavyAtomCount", "NumHAcceptors", "NumHDonors",
                "NumHeteroatoms", "NumRotatableBonds", "NumValenceElectrons", "NumAromaticRings",
                "NumSaturatedRings", "NumAliphaticRings", "RingCount", "TPSA", "LabuteASA",
                "BalabanJ", "BertzCT"]

def descriptors(smiles_list):
    mols = [Chem.MolFromSmiles(s) for s in smiles_list]
    return np.array([[getattr(Descriptors, c)(m) for c in feature_cols] for m in mols])

D_train = descriptors(train_smiles)
D_gen = descriptors(valid_pre)
scaler = StandardScaler().fit(D_train)
pca = PCA(n_components=2).fit(scaler.transform(D_train))
Z_train = pca.transform(scaler.transform(D_train))
Z_gen = pca.transform(scaler.transform(D_gen))

fig, axes = plt.subplots(1, 4, figsize=(15.5, 3.4))
for ax, col, bins in zip(axes[:3], props_train.columns, [40, 40, range(0, 8)]):
    ax.hist(props_train[col], bins=bins, density=True, alpha=0.5, label="AqSolDB (train)")
    ax.hist(props_gen[col], bins=bins, density=True, alpha=0.5, label="Generated")
    ax.set_xlabel(col); ax.set_ylabel("Density")
axes[0].legend(); axes[0].set_xlim(0, 800)

axes[3].scatter(Z_train[:, 0], Z_train[:, 1], s=4, alpha=0.3, label="AqSolDB (train)")
axes[3].scatter(Z_gen[:, 0], Z_gen[:, 1], s=6, alpha=0.6, color="tab:orange", label="Generated")
axes[3].set_xlabel("PC1"); axes[3].set_ylabel("PC2"); axes[3].legend(fontsize=8)
axes[3].set_title("PCA chemical space (17 descriptors)")
axes[3].set_xlim(-6, 12); axes[3].set_ylim(-6, 8)
plt.tight_layout()
plt.show()

### 3.2 负对数似然（NLL）：模型觉得哪个分子“更自然”？

除了采样生成新分子，训练好的语言模型还有一个极具价值的逆向用途：**给任意一个现成的分子打分，评估它在训练分布中有多“自然”**。

具体做法是：将一个给定的 SMILES 字符串喂入模型，累加每一步真实词元对应的负对数概率 $-\log P(s_t \mid s_{<t})$，得到该分子的总负对数似然（`NLL`），再除以词元数得到 **每词元平均负对数似然（`NLL per token`）**。`NLL per token` 越低，说明模型在每一步看到该分子的下一个原子时都觉得“顺理成章”；反之，如果 `NLL per token` 显著偏高，说明该分子包含训练集中罕见的化学连接方式，处于模型的**分布外（Out-of-Distribution, OOD）**区域。

我们先观察预训练模型读完前缀 `c1ccccc`（五个芳香碳加一个未闭合的环编号 `1`）后对下一个词元的预测分布，再对比 6 个代表性分子的 `NLL per token`：

In [ ]:
@torch.no_grad()
def next_token_probs(model, vocab, prefix):
    model.eval()
    x = torch.tensor([[vocab.bos] + [vocab.stoi[t] for t in tokenize(prefix)]])
    logits, _ = model(x)
    return torch.softmax(logits[0, -1], dim=-1)

p = next_token_probs(pretrained, vocab_pre, "c1ccccc")
top = torch.topk(p, 6)
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar([vocab_pre.tokens[i] for i in top.indices], top.values)
ax.set_ylabel("P(next token)")
ax.set_title("Predicted next token after prefix 'c1ccccc'")
plt.tight_layout()
plt.show()

In [ ]:
@torch.no_grad()
def nll(model, vocab, smi):
    """计算单个 SMILES 序列的总负对数似然（所有词元的 -log P 之和）。"""
    model.eval()
    x = torch.tensor([vocab.encode(smi)])
    logits, _ = model(x[:, :-1])
    return nn.functional.cross_entropy(logits[0], x[0, 1:], reduction="sum").item()

examples = {
    "benzene": "c1ccccc1",
    "toluene": "Cc1ccccc1",
    "aspirin": "CC(=O)Oc1ccccc1C(=O)O",
    "pentacene": "c1ccc2cc3cc4cc5ccccc5cc4cc3cc2c1",
    "C8-BTBT (organic semiconductor)": c8btbt,
    "perfluoro-chloro oxirane (exotic)": "FC(F)(F)C1(F)OC1(Cl)Cl",
}
rows = [{"name": k, "SMILES": v, "tokens": len(tokenize(v)), "NLL": nll(pretrained, vocab_pre, v)}
        for k, v in examples.items()]
tab = pd.DataFrame(rows)
tab["NLL per token"] = tab["NLL"] / tab["tokens"]
tab.round(2)

可以看到：阿司匹林、苯、甲苯的 `NLL per token` 在 $0.78\sim 1.07$ 之间，符合 AqSolDB 常见药物与有机化工分子的特征；而有机半导体分子 **C8-BTBT** 中稠合的苯并噻吩并噻吩（`sc1c2sc2`）骨架在通用水溶解度库中极为罕见，因此 `NLL per token` 升高到了 $1.11$；至于全氟氯代环氧乙烷这类结构怪异的分子，`NLL per token` 高达 $1.27$。

---
## 4. 跨领域迁移学习：将通用语言模型微调至有机半导体

现在进入本节的核心实战：我们的目标领域数据集 `data/osc_finetune.csv` 收录了 **159 个真实的有机半导体（Organic Semiconductors, OSC）分子**，涵盖寡聚噻吩（`oligothiophene`）、稠合噻吩（`fused thiophene`）、给体-受体共轭小分子（`donor-acceptor`）和并苯类（`acene`）。

我们先读取这批分子并可视化其中 8 个代表性结构：

In [ ]:
osc = pd.read_csv(find_file("osc_finetune.csv"))
print(osc["Family"].value_counts())
osc_smiles = [canonical(s) for s in osc["SMILES"]]
ok = [all(t in vocab_pre.stoi for t in tokenize(s)) for s in osc_smiles]
print("含预训练词表外词元的分子数:", sum(not o for o in ok))
osc_smiles = [s for s, o in zip(osc_smiles, ok) if o]

show = osc.sample(8, random_state=3)
Draw.MolsToGridImage([Chem.MolFromSmiles(s) for s in show["SMILES"]], molsPerRow=4, subImgSize=(230, 150),
                     legends=show["Name"].tolist())

### 4.1 划分微调训练集与留出集（Held-out Set）

为了严格检验微调后的模型究竟是“真正学会了有机半导体家族的结构规律”，还是“仅仅背下了微调训练集里的分子”，我们先从 159 个有机半导体中随机留出 **24 个分子作为留出验证集 `osc_val`**（绝不参与微调训练），仅用其余 **135 个分子（`osc_train`）** 进行微调。

微调开始前，先测量预训练模型在 `AqSolDB 验证集` 与 `有机半导体留出集` 上的每词元 NLL：

In [ ]:
rng = np.random.default_rng(0)
perm = rng.permutation(len(osc_smiles))
osc_val = [osc_smiles[i] for i in perm[:24]]
osc_train = [osc_smiles[i] for i in perm[24:]]
print(f"微调训练集: {len(osc_train)} 个分子 | 留出验证集: {len(osc_val)} 个分子")

OSC_MAXLEN = max(len(tokenize(s)) for s in osc_smiles) + 12
def nll_per_token(model, vocab, smiles_list):
    return np.mean([nll(model, vocab, s) / (len(tokenize(s)) + 1) for s in smiles_list])

print("微调前每个词元的平均 NLL:")
print(f"  AqSolDB 验证集:            {nll_per_token(pretrained, vocab_pre, val_smiles):.3f}")
print(f"  有机半导体留出集 (osc_val): {nll_per_token(pretrained, vocab_pre, osc_val):.3f}")

### 4.2 结合 10 倍 SMILES 增强进行迁移微调

微调的代码与预训练完全共享同一个损失函数与优化步骤，只有三点工程调整：

1. **拷贝预训练权重起点**：使用 `copy.deepcopy(pretrained)` 从已经掌握了 SMILES 语法的预训练参数出发；
2. **10 倍 SMILES 增强**：由于微调集只有 135 个分子，每一轮我们将每个分子随机重写 10 遍（`n_aug=10`，每轮 1,350 条序列），防止模型死记单一规范字符串；
3. **较小学习率与适度轮数**：采用较小的学习率（`lr=5e-4`）微调 20 轮，并实时监控微调集、有机半导体留出集和原 AqSolDB 验证集三者的 `NLL per token` 变化。

In [ ]:
def randomize_smiles(smi, vocab, rng, max_tokens=OSC_MAXLEN - 2):
    mol = Chem.MolFromSmiles(smi)
    for r in Chem.MolToRandomSmilesVect(mol, 10, randomSeed=int(rng.integers(1 << 31))):
        if len(tokenize(r)) <= max_tokens and all(t in vocab.stoi for t in tokenize(r)):
            return r
    return smi

def finetune(model, smiles_list, vocab, epochs=20, lr=5e-4, n_aug=10, seed=0):
    model = copy.deepcopy(model)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    rng = np.random.default_rng(seed)
    for ep in range(epochs):
        aug = [randomize_smiles(s, vocab, rng) for s in smiles_list for _ in range(n_aug)]
        X = to_tensor(aug, vocab, maxlen=OSC_MAXLEN)
        model.train()
        perm = torch.randperm(len(X))
        for i in range(0, len(X), 64):
            xb = X[perm[i : i + 64]]
            logits, _ = model(xb[:, :-1])
            loss = loss_fn(logits.reshape(-1, len(vocab)), xb[:, 1:].reshape(-1))
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        if (ep + 1) % 5 == 0:
            print(f"Epoch {ep+1:2d} | 微调训练集 NLL/tok: {nll_per_token(model, vocab, smiles_list):.3f} | "
                  f"OSC 留出集: {nll_per_token(model, vocab, osc_val):.3f} | "
                  f"AqSolDB 验证集: {nll_per_token(model, vocab, val_smiles):.3f}")
    return model

set_seed(0)
finetuned = finetune(pretrained, osc_train, vocab_pre, epochs=20)

这三列数字清晰地揭示了迁移学习的动态规律：

- **微调训练集 NLL** 从 $1.2+$ 降至 $0.55$：模型正在快速吸收这 135 个有机半导体的结构模式；
- **OSC 留出验证集 NLL** 从 $1.236$ 同步降至 $0.837$：从未参与微调的 24 个有机半导体分子在模型眼里也变得越来越“自然”，证明模型学到的是整个有机半导体家族的共轭构筑规律，而非简单背诵；
- **AqSolDB 验证集 NLL** 从 $0.979$ 上升至 $1.850$：随着概率分布向噻吩与稠环体系集中，模型开始逐渐遗忘原来的通用药物与脂肪族小分子，这就是深度学习中著名的**灾难性遗忘（Catastrophic Forgetting）**。因此微调轮数不宜过多，一旦留出集 NLL 趋于平缓就应及时停止（Early Stopping）。

### 4.3 微调前后的生成分子对比

分别从微调前的 `pretrained` 和微调后的 `finetuned` 中采样 500 个分子，对比生成指标、**含噻吩环比例（`contains thiophene`）**以及**平均芳香环数（`mean aromatic rings`）**，并画出微调后生成的全新分子：

In [ ]:
thiophene = Chem.MolFromSmarts("c1ccsc1")

def describe(samples, name):
    m = evaluate_samples(samples, train_set | set(osc_smiles))
    valid = [Chem.MolFromSmiles(s) for s in samples]
    valid = [x for x in valid if x is not None]
    frac_thio = np.mean([x.HasSubstructMatch(thiophene) for x in valid])
    rings = np.mean([Descriptors.NumAromaticRings(x) for x in valid])
    return {"model": name, **{k: round(v, 2) for k, v in m.items()},
            "contains thiophene": round(frac_thio, 2), "mean aromatic rings": round(rings, 2)}

set_seed(0)
samples_before = sample(pretrained, vocab_pre, n=500)
samples_after = sample(finetuned, vocab_pre, n=500, maxlen=OSC_MAXLEN + 20)
pd.DataFrame([describe(samples_before, "pretrained (AqSolDB)"),
              describe(samples_after, "fine-tuned (OSC)")]).set_index("model")

In [ ]:
set_seed(2)
samples_vis = sample(finetuned, vocab_pre, n=16, maxlen=OSC_MAXLEN + 20)
mols_vis = [Chem.MolFromSmiles(s) for s in samples_vis]
good = [(s, m) for s, m in zip(samples_vis, mols_vis) if m is not None][:12]
Draw.MolsToGridImage([m for _, m in good], molsPerRow=4, subImgSize=(230, 160),
                     legends=["novel OSC" if canonical(s) not in set(osc_smiles) else "in fine-tune set" for s, _ in good])

仅凭借 135 个有机半导体样本和不到两分钟的微调，生成分子中含有噻吩环的比例就从 **1%** 跃升至 **84%**，平均芳香环数从 **0.82** 增加到 **2.77**，而且其中 **91%** 是微调集和预训练集中均未出现过的全新共轭分子！

---
### 练习 08b-3：诊断模型眼里最“怪”与进步最大的有机半导体分子

对 `osc_smiles` 中的全部 159 个有机半导体分子，分别计算**微调前预训练模型（`pretrained`）**和**微调后模型（`finetuned`）**的每词元平均负对数似然（`NLL / (len(tokenize(s)) + 1)`）：
1. 画出微调前 vs. 微调后的每词元 NLL 散点图（横轴微调前，纵轴微调后，用不同颜色区分参与微调的 135 个分子 `osc_train` 与留出的 24 个分子 `osc_val`，并画出 $y=x$ 对角参考线）；
2. 找出微调前 `pretrained` 眼里最“怪”（NLL/token 最高）的 3 个分子，以及微调后 NLL/token 下降幅度最大的 3 个分子，用 `Draw.MolsToGridImage` 画出来，观察它们具有什么共同的化学结构特征。

In [ ]:
# 练习 08b-3：在这里写你的代码


### 练习 08b-4：生成 $\rightarrow$ 代理模型性质筛选与适用域（Applicability Domain）检验

在工业界分子设计中，微调后的生成器通常作为“候选分子提议器（Proposal Generator）”，与第 02 讲学过的监督学习性质预测模型串联使用。请完成以下闭环筛选实验：

1. 使用微调后的模型 `finetuned` 采样 2000 个分子（`maxlen=OSC_MAXLEN + 20`），过滤保留合法、去重、且不在 `osc_smiles` 中的新分子；
2. 在 AqSolDB 训练集 `D_train`（第 3.1 节已算好的 17 个 RDKit 描述符）和对应溶解度目标值上训练一个随机森林回归器 `RandomForestRegressor(n_estimators=200, random_state=0)`；
3. 预测这批新生成的有机半导体分子的水溶解度 $\log S$，画出预测 $\log S$ 的分布直方图，并把**含有噻吩环**的新分子中预测溶解度最高的 6 个分子画出来；
4. **思考与检验**：结合第 `03a` 讲的**模型适用域（Applicability Domain）**概念，把这批微调后生成的有机半导体投影到第 3.1 节的 AqSolDB PCA 空间中，判断在 AqSolDB 上训练的随机森林对疏水大共轭有机半导体的预测是否可靠？

In [ ]:
# 练习 08b-4：在这里写你的代码


---
## 5. 总结与演进：从无条件语言模型到条件生成与 Transformer

通过 `08a` 和 `08b`，我们已经掌握了基于循环神经网络（LSTM）的分子序列生成全流程。在实际材料与药物设计中，若想让序列模型从“模仿已有数据分布”走向“定向设计目标材料或预测化学反应”，主要有三条技术路线：

| 技术路线 | 核心思想 | 代表工作与本课程对应章节 |
|---|---|---|
| **迁移微调（Transfer Fine-Tuning）** | 在大通用库上预训练语法，在百余个目标领域好分子上结合 SMILES 增强微调 | Segler et al. (*ACS Cent. Sci.* 2018)；本节第 4 节 |
| **强化学习与潜空间优化（RL / BO）** | 生成分子 $\rightarrow$ 代理模型打分 $\rightarrow$ 奖励高分分子或沿潜空间梯度搜索 | REINVENT (Olivecrona 2017)；第 `09a` / `09b` 讲将深入实战 |
| **序列到序列条件生成（Seq2Seq / Transformer）** | 将反应物序列或目标条件编码为上下文，通过**注意力机制（Attention）**直接解码生成产物分子 | Molecular Transformer (Schwaller 2019)；下一节 `08c` |

在下一节 **`08c_reaction_transformer.ipynb`** 中，我们将告别一步步串行传递隐状态的 LSTM，亲手用 PyTorch 从零实现现代大语言模型与化学反应预测的核心架构——**缩放点积注意力（Scaled Dot-Product Attention）与迷你 Transformer**，并在有机半导体经典的 **Suzuki-Miyaura 交叉偶联反应数据集（`data/suzuki_toy.csv`）** 与真实专利反应样本（`data/uspto_test_sample.csv`）上实现端到端的化学反应产物预测。